![image.png](https://raw.githubusercontent.com/ambideXtrous9/Finetune-Qwen3-using-Unsloth/refs/heads/main/Experimental/GemmaUnsloth.png)

## **🦥 Install Packages**

In [1]:
!nvidia-smi

Sun Dec 21 17:53:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -qq -U evaluate rouge_score

In [59]:
from transformers import set_seed

SEED = 42
set_seed(SEED)


In [3]:
import unsloth
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-21 17:54:17.234858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766339657.669031     116 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766339657.787637     116 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766339658.869145     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766339658.869182     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766339658.869185     116 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

In [5]:
max_seq_length = 1024

## **🦥 Loading Gemma3 Model**

In [6]:
# Load the base model and tokenizer

model_name = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,   # Define context length
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # Add your token if using a gated model
)

==((====))==  Unsloth 2025.12.8: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [7]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # LoRA rank (higher rank = more parameters, potentially better fit but more memory)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", # Target attention and MLP layers
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Scaling factor (often set to r or 2*r)
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none",    # Fine-tuning bias terms ('none' is often optimal)
    # Use Unsloth's gradient checkpointing for memory saving
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False, # Rank Stable LoRA (optional)
    loftq_config = None, # LoftQ initialization (optional)
)

Unsloth: Making `model.base_model.model.model` require gradients


In [8]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

## **🦥 Loading BengaliChat Data**

In [9]:
from datasets import load_dataset

bengali_dataset = load_dataset("rishiraj/bengalichat")


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a3cbd3fa0a5ef2(…):   0%|          | 0.00/26.3M [00:00<?, ?B/s]

data/test-00000-of-00001-3b2e30253805f2b(…):   0%|          | 0.00/1.42M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

## **🦥 Train Dataset**

In [10]:
traindata = bengali_dataset["train"]

In [11]:
traindata[100]

{'prompt': 'Who are the members of Kero Kero Bonito, and what was their first debut studio album release?',
 'prompt_id': '7fd88d70fca07c43ca6e439d3e9a65a7ce3234fffb5bbbc2137d19bffa46c793',
 'messages': [{'content': '', 'role': 'system'},
  {'content': 'কেরো কেরো বোনিটোর সদস্য কারা এবং তাদের প্রথম ডেবিউ স্টুডিও অ্যালবাম রিলিজটি কী ছিল?',
   'role': 'user'},
  {'content': 'কেরো কেরো বোনিটোতে ব্যান্ড সদস্য গুস লোব্বান, জেমি বুলেড এবং সারা মিডোরি পেরি নিয়ে গঠিত।ইন্ডি পপ ত্রয়ী 2013 সালে তাদের প্রথম মিক্সটেক, ইন্ট্রো বোনিটো রেখেছিল এবং ডাবল ডেনিম রেকর্ডসের মাধ্যমে 2016 সালে তাদের অফিসিয়াল ডেবিউ স্টুডিও অ্যালবাম রিলিজ, বোনিটো জেনারেশন করেছে।',
   'role': 'assistant'}],
 'category': 'Open QA',
 'text': '<|system|>\n</s>\n<|user|>\nকেরো কেরো বোনিটোর সদস্য কারা এবং তাদের প্রথম ডেবিউ স্টুডিও অ্যালবাম রিলিজটি কী ছিল?</s>\n<|assistant|>\nকেরো কেরো বোনিটোতে ব্যান্ড সদস্য গুস লোব্বান, জেমি বুলেড এবং সারা মিডোরি পেরি নিয়ে গঠিত।ইন্ডি পপ ত্রয়ী 2013 সালে তাদের প্রথম মিক্সটেক, ইন্ট্রো বোনিটো রেখেছিল

In [12]:
def convert_to_minimal_infini_format(examples):
    conversations = []
    sources = []
    scores = []

    for msgs in examples["messages"]:
        ua = [m for m in msgs if m["role"] in ("user", "assistant")]

        if len(ua) == 2 and ua[0]["role"] == "user":
            conversations.append([
                {"role": "user", "content": ua[0]["content"]},
                {"role": "assistant", "content": ua[1]["content"]},
            ])
            sources.append("bengalichat-openqa")
            scores.append(1.0)
        else:
            conversations.append(None)
            sources.append(None)
            scores.append(None)

    return {
        "conversations": conversations,
        "source": sources,
        "score": scores,
    }


In [13]:
dataset = traindata.map(
    convert_to_minimal_infini_format,
    batched=True,
    remove_columns=traindata.column_names
)

dataset = dataset.filter(lambda x: x["conversations"] is not None)


Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9500 [00:00<?, ? examples/s]

In [14]:
dataset[100]

{'conversations': [{'content': 'আমি বাচ্চা করছি, এবং আমার কারুশিল্পের কমপক্ষে 3 টি ধারণা প্রয়োজন।উপকরণগুলি অবশ্যই আমি ইয়ার্ডে খুঁজে পেতে পারি পরিবারের উপকরণ বা স্টাফ হতে হবে।জিনিসগুলিকে একসাথে রাখার জন্য আমার কাছে কেবল আঠালো এবং টেপ রয়েছে।আমার সাজসজ্জার জন্য পেইন্ট আছে।',
   'role': 'user'},
  {'content': "কোনও সমস্যা নেই, এখানে তিনটি ছাগলছানা-বান্ধব নৈপুণ্য আইডিয়া রয়েছে যা আপনি বাড়ির চারপাশে আইটেমগুলি দিয়ে তৈরি করতে পারেন:\n\n1. আঁকা শিলা: এই নৈপুণ্য সবই সৃজনশীলতা সম্পর্কে!বাচ্চারা যা চায় তা আঁকতে পারে বা আপনি কোনও থিমের সিদ্ধান্ত নিতে পারেন এবং তার উপর ভিত্তি করে শিলাগুলি আঁকতে পারেন।\n২. পেপার টিউব প্রাণী: আপনি এটির জন্য খালি টয়লেট পেপার বা কাগজের তোয়ালে টিউব ব্যবহার করতে পারেন।অবজেক্টটি হ'ল আপনার টিউবটিকে একটি ছোট প্রাণীর মতো দেখায়;আপনি উপরে কান কেটে ফেলতে পারেন এবং চেহারাটি সম্পূর্ণ করতে ছোট মুখ, পা ইত্যাদি আঁকতে পারেন।\n৩. ম্যাকারনি গহনা: কিছু পাস্তা নুডলস এবং স্ট্রিং দিয়ে আপনার নিজস্ব ফ্যাশন লাইন তৈরি করুন!নিশ্চিত হয়ে নিন যে আপনি নুডলস ব্যবহার করেছেন যা তাদের মাধ্যম

In [15]:
from unsloth.chat_templates import standardize_data_formats

In [16]:
dataset = standardize_data_formats(dataset)

Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/8705 [00:00<?, ? examples/s]

In [17]:
dataset[100]

{'conversations': [{'content': 'আমি বাচ্চা করছি, এবং আমার কারুশিল্পের কমপক্ষে 3 টি ধারণা প্রয়োজন।উপকরণগুলি অবশ্যই আমি ইয়ার্ডে খুঁজে পেতে পারি পরিবারের উপকরণ বা স্টাফ হতে হবে।জিনিসগুলিকে একসাথে রাখার জন্য আমার কাছে কেবল আঠালো এবং টেপ রয়েছে।আমার সাজসজ্জার জন্য পেইন্ট আছে।',
   'role': 'user'},
  {'content': "কোনও সমস্যা নেই, এখানে তিনটি ছাগলছানা-বান্ধব নৈপুণ্য আইডিয়া রয়েছে যা আপনি বাড়ির চারপাশে আইটেমগুলি দিয়ে তৈরি করতে পারেন:\n\n1. আঁকা শিলা: এই নৈপুণ্য সবই সৃজনশীলতা সম্পর্কে!বাচ্চারা যা চায় তা আঁকতে পারে বা আপনি কোনও থিমের সিদ্ধান্ত নিতে পারেন এবং তার উপর ভিত্তি করে শিলাগুলি আঁকতে পারেন।\n২. পেপার টিউব প্রাণী: আপনি এটির জন্য খালি টয়লেট পেপার বা কাগজের তোয়ালে টিউব ব্যবহার করতে পারেন।অবজেক্টটি হ'ল আপনার টিউবটিকে একটি ছোট প্রাণীর মতো দেখায়;আপনি উপরে কান কেটে ফেলতে পারেন এবং চেহারাটি সম্পূর্ণ করতে ছোট মুখ, পা ইত্যাদি আঁকতে পারেন।\n৩. ম্যাকারনি গহনা: কিছু পাস্তা নুডলস এবং স্ট্রিং দিয়ে আপনার নিজস্ব ফ্যাশন লাইন তৈরি করুন!নিশ্চিত হয়ে নিন যে আপনি নুডলস ব্যবহার করেছেন যা তাদের মাধ্যম

In [18]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/8705 [00:00<?, ? examples/s]

In [19]:
dataset[100]["text"]

"<start_of_turn>user\nআমি বাচ্চা করছি, এবং আমার কারুশিল্পের কমপক্ষে 3 টি ধারণা প্রয়োজন।উপকরণগুলি অবশ্যই আমি ইয়ার্ডে খুঁজে পেতে পারি পরিবারের উপকরণ বা স্টাফ হতে হবে।জিনিসগুলিকে একসাথে রাখার জন্য আমার কাছে কেবল আঠালো এবং টেপ রয়েছে।আমার সাজসজ্জার জন্য পেইন্ট আছে।<end_of_turn>\n<start_of_turn>model\nকোনও সমস্যা নেই, এখানে তিনটি ছাগলছানা-বান্ধব নৈপুণ্য আইডিয়া রয়েছে যা আপনি বাড়ির চারপাশে আইটেমগুলি দিয়ে তৈরি করতে পারেন:\n\n1. আঁকা শিলা: এই নৈপুণ্য সবই সৃজনশীলতা সম্পর্কে!বাচ্চারা যা চায় তা আঁকতে পারে বা আপনি কোনও থিমের সিদ্ধান্ত নিতে পারেন এবং তার উপর ভিত্তি করে শিলাগুলি আঁকতে পারেন।\n২. পেপার টিউব প্রাণী: আপনি এটির জন্য খালি টয়লেট পেপার বা কাগজের তোয়ালে টিউব ব্যবহার করতে পারেন।অবজেক্টটি হ'ল আপনার টিউবটিকে একটি ছোট প্রাণীর মতো দেখায়;আপনি উপরে কান কেটে ফেলতে পারেন এবং চেহারাটি সম্পূর্ণ করতে ছোট মুখ, পা ইত্যাদি আঁকতে পারেন।\n৩. ম্যাকারনি গহনা: কিছু পাস্তা নুডলস এবং স্ট্রিং দিয়ে আপনার নিজস্ব ফ্যাশন লাইন তৈরি করুন!নিশ্চিত হয়ে নিন যে আপনি নুডলস ব্যবহার করেছেন যা তাদের মাধ্যমে স্ট্রিং থ

## **🦥 Validation Dataset**

In [20]:
valset = bengali_dataset["test"].map(
    convert_to_minimal_infini_format,
    batched=True,
    remove_columns=bengali_dataset["test"].column_names
)

valset = valset.filter(lambda x: x["conversations"] is not None)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

In [21]:
valset = standardize_data_formats(valset)

Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/446 [00:00<?, ? examples/s]

In [22]:
valset = valset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/446 [00:00<?, ? examples/s]

In [23]:
valset[100]["text"]

'<start_of_turn>user\nআপনি যদি দুধের জন্য মুদি দোকানে ভ্রমণে যান এমন কোনও মহিলার সম্পর্কে একটি ছোট গল্প লিখেন তবে আমি এটি চাই।দয়া করে 200 টিরও কম শব্দ তৈরি করুন এবং শেষে একটি চমক তৈরি করুন।<end_of_turn>\n<start_of_turn>model\nমারিয়া রেফ্রিজারেটরে তাকিয়ে দীর্ঘশ্বাস ফেলল যখন সে দেখল যে পরিবারটি আবার দুধের বাইরে চলে গেছে।তিনি জানতেন যে তার বাচ্চাদের সকালে প্রাতঃরাশের জন্য দুধের প্রয়োজন ছিল এবং যদি তিনি এখনই দোকানে না যান তবে এটি সবার জন্য একটি বড় সমস্যা সৃষ্টি করবে।তিনি বাড়ির চারপাশে তাকালেন এবং দরজাটি বের করার আগে তার চাবিগুলি এবং তার মানিব্যাগটি পেয়েছিলেন।\n\nমারিয়া যখন স্টোরের সামনের দিকে টানল তখন সে দেখে হতবাক হয়ে গেল যে লাইটগুলি চালু আছে তবে কেউ ভবনে নেই।তিনি নির্জন আইলগুলির মধ্য দিয়ে সাবধানে হাঁটলেন, সবাই কোথায় গিয়েছিল সে সম্পর্কে ক্লু খুঁজছিলেন।হঠাৎ করেই, তিনি তার পিছনে আন্দোলন শুনেছিলেন।তিনি দ্রুত ঘুরে দাঁড়ালেন, লড়াইয়ের জন্য প্রস্তুত।\n\nতবে তার পিছনে কোনও ব্যক্তিকে খুঁজে পাওয়ার পরিবর্তে তিনি দেখলেন সিরিয়াল আইলের মাঝখানে একটি বড় দুধ গরু দাঁড়িয়ে আছে।গরু জোরে জোর

## **🦥 Training**

In [24]:
from trl import SFTTrainer, SFTConfig

In [25]:
sftconfig = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 8, # Effective batch size = 2 * 4 = 8
        warmup_steps = 5,
        max_steps = 30,                 # Short run for demonstration; set to None for full epochs
        # num_train_epochs = 1,         # Alternatively, train for 1 full epoch
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), # Use bf16 if available, else fp16
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",           # Use 8-bit AdamW optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        per_device_eval_batch_size=8,
        seed = 42,
        dataloader_pin_memory=True, #fast gpu data transfer
        output_dir = "outputs",         # Directory to save checkpoints
        report_to = "none",             # Disable external reporting
        eval_strategy="steps",  # Evaluate during training
        eval_steps=5,                 # Evaluate every 5 steps
        fp16_full_eval = True,
        eval_accumulation_steps=1,
        load_best_model_at_end=True, # Load best model based on evaluation metric
        metric_for_best_model="eval_loss",  
        greater_is_better=False,           # For accuracy, higher is better
        dataset_num_proc=1

    )

In [26]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = valset, # Can set up evaluation!
    args = sftconfig
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/8705 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/446 [00:00<?, ? examples/s]

In [27]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=8):   0%|          | 0/8705 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/446 [00:00<?, ? examples/s]

In [28]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<bos><start_of_turn>user\nআমি বাচ্চা করছি, এবং আমার কারুশিল্পের কমপক্ষে 3 টি ধারণা প্রয়োজন।উপকরণগুলি অবশ্যই আমি ইয়ার্ডে খুঁজে পেতে পারি পরিবারের উপকরণ বা স্টাফ হতে হবে।জিনিসগুলিকে একসাথে রাখার জন্য আমার কাছে কেবল আঠালো এবং টেপ রয়েছে।আমার সাজসজ্জার জন্য পেইন্ট আছে।<end_of_turn>\n<start_of_turn>model\nকোনও সমস্যা নেই, এখানে তিনটি ছাগলছানা-বান্ধব নৈপুণ্য আইডিয়া রয়েছে যা আপনি বাড়ির চারপাশে আইটেমগুলি দিয়ে তৈরি করতে পারেন:\n\n1. আঁকা শিলা: এই নৈপুণ্য সবই সৃজনশীলতা সম্পর্কে!বাচ্চারা যা চায় তা আঁকতে পারে বা আপনি কোনও থিমের সিদ্ধান্ত নিতে পারেন এবং তার উপর ভিত্তি করে শিলাগুলি আঁকতে পারেন।\n২. পেপার টিউব প্রাণী: আপনি এটির জন্য খালি টয়লেট পেপার বা কাগজের তোয়ালে টিউব ব্যবহার করতে পারেন।অবজেক্টটি হ'ল আপনার টিউবটিকে একটি ছোট প্রাণীর মতো দেখায়;আপনি উপরে কান কেটে ফেলতে পারেন এবং চেহারাটি সম্পূর্ণ করতে ছোট মুখ, পা ইত্যাদি আঁকতে পারেন।\n৩. ম্যাকারনি গহনা: কিছু পাস্তা নুডলস এবং স্ট্রিং দিয়ে আপনার নিজস্ব ফ্যাশন লাইন তৈরি করুন!নিশ্চিত হয়ে নিন যে আপনি নুডলস ব্যবহার করেছেন যা তাদের মাধ্যমে স্ট্

In [29]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                         কোনও সমস্যা নেই, এখানে তিনটি ছাগলছানা-বান্ধব নৈপুণ্য আইডিয়া রয়েছে যা আপনি বাড়ির চারপাশে আইটেমগুলি দিয়ে তৈরি করতে পারেন:\n\n1. আঁকা শিলা: এই নৈপুণ্য সবই সৃজনশীলতা সম্পর্কে!বাচ্চারা যা চায় তা আঁকতে পারে বা আপনি কোনও থিমের সিদ্ধান্ত নিতে পারেন এবং তার উপর ভিত্তি করে শিলাগুলি আঁকতে পারেন।\n২. পেপার টিউব প্রাণী: আপনি এটির জন্য খালি টয়লেট পেপার বা কাগজের তোয়ালে টিউব ব্যবহার করতে পারেন।অবজেক্টটি হ'ল আপনার টিউবটিকে একটি ছোট প্রাণীর মতো দেখায়;আপনি উপরে কান কেটে ফেলতে পারেন এবং চেহারাটি সম্পূর্ণ করতে ছোট মুখ, পা ইত্যাদি আঁকতে পারেন।\n৩. ম্যাকারনি গহনা: কিছু পাস্তা নুডলস এবং স্ট্রিং দিয়ে আপনার নিজস্ব ফ্যাশন লাইন তৈরি করুন!নিশ্চিত হয়ে নিন যে আপনি নুডলস ব্যবহার করেছেন যা তাদের মাধ্যমে স্ট্রিং থ্রেডযুক্ত থাকতে পারে এবং আপনার স্টাইলের সাথে ফিট করার জন্য নুডলগুলি আঁকতে পারে!<end_of_turn>\n"

In [30]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,705 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 13,045,760 of 1,012,931,712 (1.29% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
5,4.107100,3.860965
10,3.858800,4.045285
15,4.314600,4.284910
20,4.395000,4.505536
25,4.647700,4.540140
30,4.331800,4.487087


Unsloth: Not an error, but Gemma3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## **🦥 Streaming Inference**

In [31]:
from transformers import TextStreamer

In [54]:
traindata[50]

{'prompt': 'Can you please give me a poem about watching what I eat? Name it \'Healthier You". ',
 'prompt_id': '5e6ebd79b9156cb86782c664657cabfac61e11c06a938811e64e49edc3b2139d',
 'messages': [{'content': '', 'role': 'system'},
  {'content': "আমি কি খাই তা দেখার বিষয়ে আমাকে দয়া করে একটি কবিতা দিতে পারেন?এর নাম দিন 'আপনি স্বাস্থ্যকর'।",
   'role': 'user'},
  {'content': 'আপনি স্বাস্থ্যকর\n\nকার্বস, সুগার এবং ফ্যাট\nশেষ থেকে দূরে থাকুন\n\nএটি এখনও এগিয়ে যাওয়া কঠিন হতে পারে\nযখন আমরা আমাদের পথে সেট করা হয়\n\nতবে স্বাস্থ্যকর অভ্যাসগুলি আমাদের একটি স্বাস্থ্যকর জীবন দেয়\nশক্তি এবং মজা পূর্ণ\n\nআরও প্রোটিন, ফল এবং ভেজি খান\nআপনি আগের চেয়ে শক্তিশালী হয়ে উঠবেন\n\nতবে ভুলে যাবেন না আপনি যা খান তা সবই নয়\nএটি সংযম এবং যাত্রা সম্পর্কে\n\nতাই নতুন কিছু শুরুতে স্বাগতম\nআপনি একজন স্বাস্থ্যকর হয়ে ওঠার যাত্রা',
   'role': 'assistant'}],
 'category': 'Generation',
 'text': "<|system|>\n</s>\n<|user|>\nআমি কি খাই তা দেখার বিষয়ে আমাকে দয়া করে একটি কবিতা দিতে পারেন?এর নাম দিন 'আপনি স্বাস্থ্যকর

In [55]:

query = "আমি কি খাই তা দেখার বিষয়ে আমাকে দয়া করে একটি কবিতা দিতে পারেন?এর নাম দিন 'আপনি স্বাস্থ্যকর'।"

messages = [
    {"role" : "user", "content" :  query}
]

In [62]:
# Format the prompt, explicitly DISABLING thinking mode
text_input_no_think = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Crucial for generation
    enable_thinking = False,      # *** Disable thinking ***
)


print("\n--- Non-Thinking Inference ---\n")
print("Formatted Input:\n", text_input_no_think)
print("\n--- Non-Thinking Inference ---\n")


# Generate response using parameters suitable for non-thinking/chat
print("\n-----------------------------\n")
inputs = tokenizer(text_input_no_think, return_tensors = "pt").to("cuda")
streamer_no_think = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    **inputs,
    max_new_tokens = 200,
    temperature = 0.1, # Recommended for chat
    top_p = 0.8,       # Recommended for chat
    top_k = 64,
    repetition_penalty=1.15,
    streamer = streamer_no_think,
    eos_token_id = tokenizer.eos_token_id # Ensure generation stops properly
)
print("\n-----------------------------\n")


--- Non-Thinking Inference ---

Formatted Input:
 <bos><start_of_turn>user
আমি কি খাই তা দেখার বিষয়ে আমাকে দয়া করে একটি কবিতা দিতে পারেন?এর নাম দিন 'আপনি স্বাস্থ্যকর'।<end_of_turn>
<start_of_turn>model


--- Non-Thinking Inference ---


-----------------------------

এখানে আপনার 健康 সম্পর্কে কিছু তথ্য রয়েছে যা আপনাকে সহায়তা করতে পারে:

খাই মিষ্টি খাবার গ্রহণ করার সময় তার খাদ্য পরিকল্পনা এবং জীবনযাত্রার সাথে সম্পর্কিত বিভিন্ন সমস্যা থেকে রক্ষা পেতে有助于। আপনি যদি চান তবে এটি কী প্রয়োজন হবে সে সম্পর্কে আরও ভাল ধারণা получить க்கு অতিরিক্ত টিপস দেওয়া যেতে পারে, যেমন সঠিক आहार অনুসরণ করা বা ডায়েট ট্র্যাক খাওয়া। এইগুলি सभी खाद्य অঞ্চলে অন্তর্ভুক্ত করুন কারণ 食品 গ্রহণের জন্য কোনও চিন্তা না hacer करने से आपके स्वास्थ्य पर बहुत अधिक प्रभाव पड़ेगा। यदि आप अपने जीवन में सुधार करना चाहते हैं या किसी विशेष क्षेत्र के बारे में कोई चिंता है जो आपको क्या विचार कर रहा है यह जानने की आवश्यकता हो सकती है कि 如何 पोषण प्राप्त करें और स्वस्थ रहना कैसे होगा। हमेशा उन लोगों को खोजने का प्रयास करें जिनके